### Técnica XGBoost

**Carga de datos desde MinIO**

In [ ]:
import os
import sys
import pandas as pd
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.common.minio_client import download_df_parquet

ruta_raiz = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ruta_raiz not in sys.path:
    sys.path.insert(0, ruta_raiz)


access_key = os.getenv("MINIO_ACCESS_KEY")
if access_key is None:
    raise AssertionError("MINIO_ACCESS_KEY no definida.")

secret_key = os.getenv("MINIO_SECRET_KEY")
if secret_key is None:
    raise AssertionError("MINIO_SECRET_KEY no definida.")


meses = ['01', '02']
dfs = []

for mes in meses:
    try:
        s = download_df_parquet(
            access_key=access_key,
            secret_key=secret_key,
            object_name=f"grupo5/final/year=2025/month={mes}/dataset_final.parquet",
        )
        dfs.append(s)
    except Exception as e:
        print(f"  Sin datos para mes {mes}: {e}")

df = pd.concat(dfs, ignore_index=True)

print(f"¡Descarga exitosa! {len(meses)} meses")
print(f"  df: {len(df):,} filas")


**Creación de nuevas features para una ventana de tiempo más amplia**

In [ ]:
#Ordenamos por línea, parada y timestamp
df = df.sort_values(['route_id', 'stop_id', 'merge_time'])
grupo = ['route_id', 'stop_id']

#Creación de nuevas variables de delay para poder ampliar la ventana de observaciones a 30 minutos aproximadamente
df['lagged_delay_3'] = df.groupby(grupo)['delay_seconds'].shift(3)
df['lagged_delay_4'] = df.groupby(grupo)['delay_seconds'].shift(4)

#Tendencia en el retraso
df['delay_trend'] = df['lagged_delay_1'] - df['lagged_delay_4']

#Inestabilidad reciente de la línea
df['delay_std'] = df.groupby(grupo)['delay_seconds'].transform(
    lambda x: x.shift(1).rolling(4).std()
)

cols_lag = ['lagged_delay_1', 'lagged_delay_2', 'lagged_delay_3', 'lagged_delay_4']
df[cols_lag] = df[cols_lag].fillna(0)

**Selección de features y target**

In [ ]:
#Seleccion de variables
features = [
    'delay_seconds', 'lagged_delay_1', 'lagged_delay_2',
    'lagged_delay_3', 'lagged_delay_4', 'delay_trend',
    'route_rolling_delay', 'actual_headway_seconds',
    'is_unscheduled', 'num_updates', 'scheduled_time_to_end',
    'stops_to_end', 'route_id', 'direction', 'delay_std',
    'hour_sin', 'hour_cos', 'dow', 'is_weekend',
    'n_eventos_afectando', 'tipo_referente',
    'afecta_previo', 'afecta_durante', 'afecta_despues',
    'temp_extreme', 'category',
]
target = 'alert_in_next_15m'


# Eliminar filas sin target
df = df.dropna(subset=[target]).copy()
df[target] = df[target].astype(int)

**Filtro que elimina paradas que no tienen alerta en 15m pero su comportamiento puede estar alterado por alerta en 30m**

In [ ]:
mask_positivos = df[target] == 1
mask_negativos_limpios = (
    df['alert_in_next_30m'] == 0            
)

df = df[mask_positivos | mask_negativos_limpios].copy()
df = df.reset_index(drop=True)

print(f"Dataset tras filtrar negativos ambiguos: {len(df):,} filas")
print(f"  Positivos: {df[target].sum():,} ({df[target].mean()*100:.1f}%)")
print(f"  Negativos: {(df[target]==0).sum():,} ({(df[target]==0).mean()*100:.1f}%)")

**División de los datos en Train-Val-Test**

In [ ]:
df_sorted = df.sort_values('merge_time')

#Division X e y
X = df_sorted[features]
y = df_sorted[target]

print(f"Features: {len(features)}")
print(f"Filas:    {len(X):,}")
print(f"\nDistribución del target:")
print(y.value_counts(normalize=True).round(3))


#División de los datos en Entrenamiento-Validación-Test

dias = df_sorted['merge_time'].dt.date.unique()
dias_ordenados = sorted(dias)

total_dias = len(dias_ordenados)
corte_70   = dias_ordenados[int(total_dias * 0.70)]
corte_85   = dias_ordenados[int(total_dias * 0.85)]

print(f"Total días: {total_dias}")
print(f"Primer día: {dias_ordenados[0]}")
print(f"Último día: {dias_ordenados[-1]}")
print(f"\nCorte train (70%): {corte_70}")
print(f"Corte val   (85%): {corte_85}")


train = df_sorted[df_sorted['merge_time'].dt.date <  corte_70]
val   = df_sorted[(df_sorted['merge_time'].dt.date >= corte_70) &
           (df_sorted['merge_time'].dt.date <  corte_85)]
test  = df_sorted[df_sorted['merge_time'].dt.date >= corte_85]

X_train, y_train = train[features], train[target]
X_val,   y_val   = val[features],   val[target]
X_test,  y_test  = test[features],  test[target]

n = len(df)
print(f"Train: {len(train):,} ({len(train)/n*100:.0f}%)  "
      f"{train['merge_time'].min().date()} → {train['merge_time'].max().date()}")
print(f"Val:   {len(val):,} ({len(val)/n*100:.0f}%)  "
      f"{val['merge_time'].min().date()} → {val['merge_time'].max().date()}")
print(f"Test:  {len(test):,} ({len(test)/n*100:.0f}%)  "
      f"{test['merge_time'].min().date()} → {test['merge_time'].max().date()}")


**Encoding de categorías**

In [ ]:
#Encodear categorías

from sklearn.preprocessing import OrdinalEncoder
import category_encoders as ce

cols_target_enc  = ['category', 'tipo_referente'] 
cols_ordinal_enc = ['route_id', 'direction']   

#Target Encoding
te = ce.TargetEncoder(cols=cols_target_enc, smoothing=10) 
X_train[cols_target_enc] = te.fit_transform(X_train[cols_target_enc], y_train)
X_val[cols_target_enc]   = te.transform(X_val[cols_target_enc])
X_test[cols_target_enc]  = te.transform(X_test[cols_target_enc])

#Ordinal Encoding
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[cols_ordinal_enc] = enc.fit_transform(X_train[cols_ordinal_enc])
X_val[cols_ordinal_enc]   = enc.transform(X_val[cols_ordinal_enc])
X_test[cols_ordinal_enc]  = enc.transform(X_test[cols_ordinal_enc])

print("✓ Encoding completado")

**Búsqueda de los mejores hiperparámetros empleando optuna**

In [ ]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import (average_precision_score, classification_report, roc_auc_score,
                              average_precision_score, f1_score,
                              recall_score, precision_score)

# Ratio de desbalance para scale_pos_weight
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Ratio desbalance: {ratio:.1f}:1")

def objective(trial):
    params = {
        'max_depth':          trial.suggest_int('max_depth', 3, 10),
        'min_child_weight':   trial.suggest_int('min_child_weight', 1, 100),
        'gamma':              trial.suggest_float('gamma', 0, 10),
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators':       trial.suggest_int('n_estimators', 200, 800),
        'subsample':          trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree':   trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'colsample_bylevel':  trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        'reg_alpha':          trial.suggest_float('reg_alpha', 1e-6, 100, log=True),
        'reg_lambda':         trial.suggest_float('reg_lambda', 1e-6, 100, log=True),
        'max_delta_step':     trial.suggest_float('max_delta_step', 0, 10),
        'scale_pos_weight':   ratio,
        'tree_method':        'hist',
        'eval_metric':        'aucpr',
        'early_stopping_rounds': 30,
    }
    modelo = XGBClassifier(**params, random_state=42)
    modelo.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    y_prob = modelo.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, y_prob)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=45)
print(study.best_params)

**Entrenamiento del modelo empleando los mejores parámetros**

In [ ]:
best_params = study.best_params

params_fijos = {
    'n_estimators': 500,
    'scale_pos_weight': ratio,
    'tree_method': 'hist',
    'eval_metric': 'aucpr',
    'random_state': 42
}

parametros = {**params_fijos, **best_params}

modelo_final = XGBClassifier(**parametros)
modelo_final.fit(X_train, y_train)

**Evaluacion del modelo**

In [ ]:
import numpy as np

#Evaluación del modelo

y_prob = modelo_final.predict_proba(X_test)[:, 1]

# Threshold óptimo por F1
thresholds = np.arange(0.05, 0.95, 0.01)
f1_scores  = [f1_score(y_test, (y_prob >= t).astype(int),
                        zero_division=0) for t in thresholds]
threshold_opt = thresholds[np.argmax(f1_scores)]
y_pred = (y_prob >= threshold_opt).astype(int)


print(f"Threshold óptimo: {threshold_opt:.2f}")
print(f"\nAUC-ROC : {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC  : {average_precision_score(y_test, y_prob):.4f}")
print(f"F1      : {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall  : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"\n{classification_report(y_test, y_pred)}")

**Curva Precision-Recall con area sombreada**

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import PrecisionRecallDisplay

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva PR
PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[0])
axes[0].set_title(f'PR Curve (AUC={average_precision_score(y_test, y_prob):.3f})')
axes[0].axhline(y=y_test.mean(), color='r', linestyle='--', label='Baseline')
axes[0].legend()

# Curva ROC
from sklearn.metrics import RocCurveDisplay
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1])
axes[1].set_title('ROC Curve')
plt.tight_layout()

**Importancia de features con SHAP**

In [ ]:
import shap

# Muestra aleatoria para SHAP (costoso con 1.8M filas)
sample = X_test.sample(5000, random_state=42)
explainer = shap.TreeExplainer(modelo_final)
shap_values = explainer.shap_values(sample)

shap.summary_plot(shap_values, sample, plot_type='bar')   # importancia global
shap.summary_plot(shap_values, sample)                     # dirección del efecto

**Matriz de confusión con costes**

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(8, 6))

disp = ConfusionMatrixDisplay.from_predictions(
    y_test, 
    y_pred,
    display_labels=['Sin alerta', 'Alerta'],
    cmap='Blues',
    ax=ax,
    values_format=',d'
)

fp = ((y_pred == 1) & (y_test == 0)).sum()
fn = ((y_pred == 0) & (y_test == 1)).sum()

ax.set_title(f'Falsas alarmas (FP): {fp:,} | Alertas perdidas (FN): {fn:,}', 
             pad=20,     
             fontsize=12)

plt.tight_layout() 
plt.show()

### Distribución por linea

**Carga de datos desde MinIO**

In [ ]:
import os
import sys
import pandas as pd
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.common.minio_client import download_df_parquet

ruta_raiz = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ruta_raiz not in sys.path:
    sys.path.insert(0, ruta_raiz)


access_key = os.getenv("MINIO_ACCESS_KEY")
if access_key is None:
    raise AssertionError("MINIO_ACCESS_KEY no definida.")

secret_key = os.getenv("MINIO_SECRET_KEY")
if secret_key is None:
    raise AssertionError("MINIO_SECRET_KEY no definida.")


lineas = ['1', '2']
dfs = []

for linea in lineas:
    try:
        s = download_df_parquet(
            access_key=access_key,
            secret_key=secret_key,
            object_name=f"grupo5/aggregations/lines/line={linea}/dataset_final.parquet",
        )
        dfs.append(s)
    except Exception as e:
        print(f"  Sin datos para linea {linea}: {e}")

df = pd.concat(dfs, ignore_index=True)

print(f"¡Descarga exitosa! {len(lineas)} lineas")
print(f"  df: {len(df):,} filas")


**Creación de nuevas features para una ventana de tiempo más amplia**

In [ ]:
#Ordenamos por línea, parada y timestamp
df = df.sort_values(['route_id', 'stop_id', 'merge_time'])
grupo = ['route_id', 'stop_id']

#Creación de nuevas variables de delay para poder ampliar la ventana de observaciones a 30 minutos aproximadamente
df['lagged_delay_3'] = df.groupby(grupo)['delay_seconds'].shift(3)
df['lagged_delay_4'] = df.groupby(grupo)['delay_seconds'].shift(4)

#Tendencia en el retraso
df['delay_trend'] = df['lagged_delay_1'] - df['lagged_delay_4']

#Inestabilidad reciente de la línea
df['delay_std'] = df.groupby(grupo)['delay_seconds'].transform(
    lambda x: x.shift(1).rolling(4).std()
)

#Eliminar filas con NaN en lagged features
cols_lag = ['lagged_delay_1', 'lagged_delay_2', 'lagged_delay_3', 'lagged_delay_4']
df[cols_lag] = df[cols_lag].fillna(0)

**Selección de features y target**

In [ ]:
#Seleccion de variables
features = [
    'delay_seconds', 'lagged_delay_1', 'lagged_delay_2',
    'lagged_delay_3', 'lagged_delay_4', 'delay_trend',
    'route_rolling_delay', 'actual_headway_seconds',
    'is_unscheduled', 'num_updates', 'scheduled_time_to_end',
    'stops_to_end', 'route_id', 'direction', 'delay_std',
    'hour_sin', 'hour_cos', 'dow', 'is_weekend',
    'n_eventos_afectando', 'tipo_referente',
    'afecta_previo', 'afecta_durante', 'afecta_despues',
    'temp_extreme', 'category',
]
target = 'alert_in_next_15m'


# Eliminar filas sin target
df = df.dropna(subset=[target]).copy()
df[target] = df[target].astype(int)

**Filtro que elimina paradas que no tienen alerta en 15m pero su comportamiento puede estar alterado por alerta en 30m**

In [ ]:
mask_positivos = df[target] == 1
mask_negativos_limpios = (
    df['alert_in_next_30m'] == 0            
)

df = df[mask_positivos | mask_negativos_limpios].copy()
df = df.reset_index(drop=True)

print(f"Dataset tras filtrar negativos ambiguos: {len(df):,} filas")
print(f"  Positivos: {df[target].sum():,} ({df[target].mean()*100:.1f}%)")
print(f"  Negativos: {(df[target]==0).sum():,} ({(df[target]==0).mean()*100:.1f}%)")

**División de los datos en Train-Val-Test**

In [ ]:
df_sorted = df.sort_values('merge_time')

#Division X e y
X = df_sorted[features]
y = df_sorted[target]

print(f"Features: {len(features)}")
print(f"Filas:    {len(X):,}")
print(f"\nDistribución del target:")
print(y.value_counts(normalize=True).round(3))


#División de los datos en Entrenamiento-Validación-Test

dias = df_sorted['merge_time'].dt.date.unique()
dias_ordenados = sorted(dias)

total_dias = len(dias_ordenados)
corte_70   = dias_ordenados[int(total_dias * 0.70)]
corte_85   = dias_ordenados[int(total_dias * 0.85)]

print(f"Total días: {total_dias}")
print(f"Primer día: {dias_ordenados[0]}")
print(f"Último día: {dias_ordenados[-1]}")
print(f"\nCorte train (70%): {corte_70}")
print(f"Corte val   (85%): {corte_85}")


train = df_sorted[df_sorted['merge_time'].dt.date <  corte_70]
val   = df_sorted[(df_sorted['merge_time'].dt.date >= corte_70) &
           (df_sorted['merge_time'].dt.date <  corte_85)]
test  = df_sorted[df_sorted['merge_time'].dt.date >= corte_85]

X_train, y_train = train[features], train[target]
X_val,   y_val   = val[features],   val[target]
X_test,  y_test  = test[features],  test[target]

n = len(df)
print(f"Train: {len(train):,} ({len(train)/n*100:.0f}%)  "
      f"{train['merge_time'].min().date()} → {train['merge_time'].max().date()}")
print(f"Val:   {len(val):,} ({len(val)/n*100:.0f}%)  "
      f"{val['merge_time'].min().date()} → {val['merge_time'].max().date()}")
print(f"Test:  {len(test):,} ({len(test)/n*100:.0f}%)  "
      f"{test['merge_time'].min().date()} → {test['merge_time'].max().date()}")


**Encoding de categorías**

In [ ]:
#Encodear categorías

from sklearn.preprocessing import OrdinalEncoder
import category_encoders as ce

cols_target_enc  = ['category', 'tipo_referente'] 
cols_ordinal_enc = ['route_id', 'direction']   

#Target Encoding
te = ce.TargetEncoder(cols=cols_target_enc, smoothing=10) 
X_train[cols_target_enc] = te.fit_transform(X_train[cols_target_enc], y_train)
X_val[cols_target_enc]   = te.transform(X_val[cols_target_enc])
X_test[cols_target_enc]  = te.transform(X_test[cols_target_enc])

#Ordinal Encoding
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[cols_ordinal_enc] = enc.fit_transform(X_train[cols_ordinal_enc])
X_val[cols_ordinal_enc]   = enc.transform(X_val[cols_ordinal_enc])
X_test[cols_ordinal_enc]  = enc.transform(X_test[cols_ordinal_enc])

print("✓ Encoding completado")

**Búsqueda de los mejores hiperparámetros empleando optuna**

In [ ]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import (average_precision_score, classification_report, roc_auc_score,
                              average_precision_score, f1_score,
                              recall_score, precision_score)

# Ratio de desbalance para scale_pos_weight
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Ratio desbalance: {ratio:.1f}:1")

def objective(trial):
    params = {
        'max_depth':          trial.suggest_int('max_depth', 3, 10),
        'min_child_weight':   trial.suggest_int('min_child_weight', 1, 100),
        'gamma':              trial.suggest_float('gamma', 0, 10),
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators':       trial.suggest_int('n_estimators', 200, 800),
        'subsample':          trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree':   trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'colsample_bylevel':  trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        'reg_alpha':          trial.suggest_float('reg_alpha', 1e-6, 100, log=True),
        'reg_lambda':         trial.suggest_float('reg_lambda', 1e-6, 100, log=True),
        'max_delta_step':     trial.suggest_float('max_delta_step', 0, 10),
        'scale_pos_weight':   ratio,
        'tree_method':        'hist',
        'eval_metric':        'aucpr',
        'early_stopping_rounds': 30,
    }
    modelo = XGBClassifier(**params, random_state=42)
    modelo.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    y_prob = modelo.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, y_prob)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=45)
print(study.best_params)

**Entrenamiento del modelo empleando los mejores parámetros**

In [ ]:
best_params = study.best_params

params_fijos = {
    'n_estimators': 500,
    'scale_pos_weight': ratio,
    'tree_method': 'hist',
    'eval_metric': 'aucpr',
    'random_state': 42
}

parametros = {**params_fijos, **best_params}

modelo_final = XGBClassifier(**parametros)
modelo_final.fit(X_train, y_train)

**Evaluacion del modelo**

In [ ]:
import numpy as np

#Evaluación del modelo

y_prob = modelo_final.predict_proba(X_test)[:, 1]

# Threshold óptimo por F1
thresholds = np.arange(0.05, 0.95, 0.01)
f1_scores  = [f1_score(y_test, (y_prob >= t).astype(int),
                        zero_division=0) for t in thresholds]
threshold_opt = thresholds[np.argmax(f1_scores)]
y_pred = (y_prob >= threshold_opt).astype(int)


print(f"Threshold óptimo: {threshold_opt:.2f}")
print(f"\nAUC-ROC : {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC  : {average_precision_score(y_test, y_prob):.4f}")
print(f"F1      : {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall  : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"\n{classification_report(y_test, y_pred)}")

**Curva Precision-Recall con area sombreada**

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import PrecisionRecallDisplay

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva PR
PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[0])
axes[0].set_title(f'PR Curve (AUC={average_precision_score(y_test, y_prob):.3f})')
axes[0].axhline(y=y_test.mean(), color='r', linestyle='--', label='Baseline')
axes[0].legend()

# Curva ROC
from sklearn.metrics import RocCurveDisplay
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1])
axes[1].set_title('ROC Curve')
plt.tight_layout()

**Importancia de features con SHAP**

In [ ]:
import shap

# Muestra aleatoria para SHAP (costoso con 1.8M filas)
sample = X_test.sample(5000, random_state=42)
explainer = shap.TreeExplainer(modelo_final)
shap_values = explainer.shap_values(sample)

shap.summary_plot(shap_values, sample, plot_type='bar')  
shap.summary_plot(shap_values, sample)                   

**Matriz de confusión con costes**

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(8, 6))

disp = ConfusionMatrixDisplay.from_predictions(
    y_test, 
    y_pred,
    display_labels=['Sin alerta', 'Alerta'],
    cmap='Blues',
    ax=ax,
    values_format=',d'
)

fp = ((y_pred == 1) & (y_test == 0)).sum()
fn = ((y_pred == 0) & (y_test == 1)).sum()

ax.set_title(f'Falsas alarmas (FP): {fp:,} | Alertas perdidas (FN): {fn:,}', 
             pad=20,     
             fontsize=12)

plt.tight_layout() 
plt.show()

### Agragados cada 30 min

**Carga de datos desde MinIO**

In [ ]:
import os
import sys
import pandas as pd
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.common.minio_client import download_df_parquet

ruta_raiz = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ruta_raiz not in sys.path:
    sys.path.insert(0, ruta_raiz)


access_key = os.getenv("MINIO_ACCESS_KEY")
if access_key is None:
    raise AssertionError("MINIO_ACCESS_KEY no definida.")

secret_key = os.getenv("MINIO_SECRET_KEY")
if secret_key is None:
    raise AssertionError("MINIO_SECRET_KEY no definida.")


try:
    df = download_df_parquet(
        access_key=access_key,
        secret_key=secret_key,
        object_name=f"grupo5/aggregations/DataFrameGroupedByMin=30.parquet",
    )
except Exception as e:
        print(f"  Sin datos: {e}")


print(f"  df: {len(df):,} filas")


In [ ]:
df

**Selección de features y target**

In [ ]:
#Seleccion de variables
features = [
    "delay_seconds_mean",
    "lagged_delay_1_mean",
    "lagged_delay_2_mean",
    "route_rolling_delay_mean",
    "actual_headway_seconds_mean",
    "seconds_since_last_alert_mean",
    "afecta_previo_max",
    "hour_sin_first",
    "hour_cos_first",
    "dow_first",
    "is_weekend_max",
    "is_unscheduled_max",
    "temp_extreme_max",
    "stops_to_end_mean",
    "scheduled_time_to_end_mean",
    "num_updates_sum",
    "match_key_nunique",
    "direction",
    "route_id",
]
target = 'alert_in_next_15m_max'


# Eliminar filas sin target
df = df.dropna(subset=[target]).copy()
df[target] = df[target].astype(int)

**Filtro que elimina paradas que no tienen alerta en 15m pero su comportamiento puede estar alterado por alerta en 30m**

In [ ]:
mask_positivos = df[target] == 1
mask_negativos_limpios = (
    df['alert_in_next_30m_max'] == 0            
)

df = df[mask_positivos | mask_negativos_limpios].copy()
df = df.reset_index(drop=True)

print(f"Dataset tras filtrar negativos ambiguos: {len(df):,} filas")
print(f"  Positivos: {df[target].sum():,} ({df[target].mean()*100:.1f}%)")
print(f"  Negativos: {(df[target]==0).sum():,} ({(df[target]==0).mean()*100:.1f}%)")

**División de datos en Train-Val-Test**

In [ ]:
df_sorted = df.sort_values('merge_time')

#Division X e y
X = df_sorted[features]
y = df_sorted[target]

print(f"Features: {len(features)}")
print(f"Filas:    {len(X):,}")
print(f"\nDistribución del target:")
print(y.value_counts(normalize=True).round(3))


#División de los datos en Entrenamiento-Validación-Test

dias = df_sorted['merge_time'].dt.date.unique()
dias_ordenados = sorted(dias)

total_dias = len(dias_ordenados)
corte_70   = dias_ordenados[int(total_dias * 0.70)]
corte_85   = dias_ordenados[int(total_dias * 0.85)]

print(f"Total días: {total_dias}")
print(f"Primer día: {dias_ordenados[0]}")
print(f"Último día: {dias_ordenados[-1]}")
print(f"\nCorte train (70%): {corte_70}")
print(f"Corte val   (85%): {corte_85}")


train = df_sorted[df_sorted['merge_time'].dt.date <  corte_70]
val   = df_sorted[(df_sorted['merge_time'].dt.date >= corte_70) &
           (df_sorted['merge_time'].dt.date <  corte_85)]
test  = df_sorted[df_sorted['merge_time'].dt.date >= corte_85]

X_train, y_train = train[features], train[target]
X_val,   y_val   = val[features],   val[target]
X_test,  y_test  = test[features],  test[target]

n = len(df)
print(f"Train: {len(train):,} ({len(train)/n*100:.0f}%)  "
      f"{train['merge_time'].min().date()} → {train['merge_time'].max().date()}")
print(f"Val:   {len(val):,} ({len(val)/n*100:.0f}%)  "
      f"{val['merge_time'].min().date()} → {val['merge_time'].max().date()}")
print(f"Test:  {len(test):,} ({len(test)/n*100:.0f}%)  "
      f"{test['merge_time'].min().date()} → {test['merge_time'].max().date()}")

**Encoding de categorías**

In [ ]:
#Encodear categorías

from sklearn.preprocessing import OrdinalEncoder

cols_ordinal_enc = ['route_id', 'direction']   

#Ordinal Encoding
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[cols_ordinal_enc] = enc.fit_transform(X_train[cols_ordinal_enc])
X_val[cols_ordinal_enc]   = enc.transform(X_val[cols_ordinal_enc])
X_test[cols_ordinal_enc]  = enc.transform(X_test[cols_ordinal_enc])

print("✓ Encoding completado")

**Búsqueda de los mejores hiperparámetros empleando optuna**

In [ ]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import (average_precision_score, classification_report, roc_auc_score,
                              average_precision_score, f1_score,
                              recall_score, precision_score)

# Ratio de desbalance para scale_pos_weight
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Ratio desbalance: {ratio:.1f}:1")

def objective(trial):
    params = {
        'max_depth':          trial.suggest_int('max_depth', 3, 10),
        'min_child_weight':   trial.suggest_int('min_child_weight', 1, 100),
        'gamma':              trial.suggest_float('gamma', 0, 10),
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators':       trial.suggest_int('n_estimators', 200, 800),
        'subsample':          trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree':   trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'colsample_bylevel':  trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        'reg_alpha':          trial.suggest_float('reg_alpha', 1e-6, 100, log=True),
        'reg_lambda':         trial.suggest_float('reg_lambda', 1e-6, 100, log=True),
        'max_delta_step':     trial.suggest_float('max_delta_step', 0, 10),
        'scale_pos_weight':   ratio,
        'tree_method':        'hist',
        'eval_metric':        'aucpr',
        'early_stopping_rounds': 30,
    }
    modelo = XGBClassifier(**params, random_state=42)
    modelo.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    y_prob = modelo.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, y_prob)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(study.best_params)

**Entrenamiento del modelo empleando los mejores parámetros**

In [ ]:
best_params = study.best_params

params_fijos = {
    'n_estimators': 500,
    'scale_pos_weight': ratio,
    'tree_method': 'hist',
    'eval_metric': 'aucpr',
    'random_state': 42
}

parametros = {**params_fijos, **best_params}

modelo_final = XGBClassifier(**parametros)
modelo_final.fit(X_train, y_train)

**Evaluacion del modelo**

In [ ]:
import numpy as np

#Evaluación del modelo

y_prob = modelo_final.predict_proba(X_test)[:, 1]

# Threshold óptimo por F1
thresholds = np.arange(0.05, 0.95, 0.01)
f1_scores  = [f1_score(y_test, (y_prob >= t).astype(int),
                        zero_division=0) for t in thresholds]
threshold_opt = thresholds[np.argmax(f1_scores)]
y_pred = (y_prob >= threshold_opt).astype(int)


print(f"Threshold óptimo: {threshold_opt:.2f}")
print(f"\nAUC-ROC : {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC  : {average_precision_score(y_test, y_prob):.4f}")
print(f"F1      : {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall  : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"\n{classification_report(y_test, y_pred)}")

**Curva Precision-Recall con area sombreada**

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import PrecisionRecallDisplay

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva PR
PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[0])
axes[0].set_title(f'PR Curve (AUC={average_precision_score(y_test, y_prob):.3f})')
axes[0].axhline(y=y_test.mean(), color='r', linestyle='--', label='Baseline')
axes[0].legend()

# Curva ROC
from sklearn.metrics import RocCurveDisplay
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1])
axes[1].set_title('ROC Curve')
plt.tight_layout()

**Importancia de features con SHAP**

In [ ]:
import shap

# Muestra aleatoria para SHAP (costoso con 1.8M filas)
sample = X_test.sample(5000, random_state=42)
explainer = shap.TreeExplainer(modelo_final)
shap_values = explainer.shap_values(sample)

shap.summary_plot(shap_values, sample, plot_type='bar')   # importancia global
shap.summary_plot(shap_values, sample)                     # dirección del efecto

**Matriz de confusión con costes**

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(8, 6))

disp = ConfusionMatrixDisplay.from_predictions(
    y_test, 
    y_pred,
    display_labels=['Sin alerta', 'Alerta'],
    cmap='Blues',
    ax=ax,
    values_format=',d'
)

fp = ((y_pred == 1) & (y_test == 0)).sum()
fn = ((y_pred == 0) & (y_test == 1)).sum()

ax.set_title(f'Falsas alarmas (FP): {fp:,} | Alertas perdidas (FN): {fn:,}', 
             pad=20,     
             fontsize=12)

plt.tight_layout() 
plt.show()